In [1]:
from pathlib import Path
import tensorflow as tf
from tensorflow.keras.preprocessing.image import img_to_array, load_img
import numpy as np
import requests
from PIL import Image



In [ ]:
#pip install tensorflow
#python -m pip install Pillow  ''' run this command in terminal to install Pillow library for image processing '''


SyntaxError: invalid syntax (338903800.py, line 3)

In [2]:
DataSet_Dir = './Input'

In [3]:
import os, random

batch_size = 32

def build_dataset_no_resize(data_dir, validation_split=0.2, subset="training",
                             seed=123, batch_size=32, label_mode="categorical"):
    """Load images from a directory WITHOUT resizing or normalizing."""
    # Discover class names from sub-folder names
    class_names = sorted([
        d for d in os.listdir(data_dir)
        if os.path.isdir(os.path.join(data_dir, d))
    ])
    num_classes  = len(class_names)
    class_to_idx = {c: i for i, c in enumerate(class_names)}

    # Collect every image path + its integer label
    all_files, all_labels = [], []
    for cls in class_names:
        cls_dir = os.path.join(data_dir, cls)
        for fname in sorted(os.listdir(cls_dir)):
            if fname.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                all_files.append(os.path.join(cls_dir, fname))
                all_labels.append(class_to_idx[cls])

    # Deterministic shuffle (matches image_dataset_from_directory behaviour)
    combined = list(zip(all_files, all_labels))
    random.seed(seed)
    random.shuffle(combined)
    all_files, all_labels = zip(*combined)

    # 80 / 20 split
    n         = len(all_files)
    split_idx = int(n * (1 - validation_split))
    if subset == "training":
        files  = list(all_files[:split_idx])
        labels = list(all_labels[:split_idx])
    else:
        files  = list(all_files[split_idx:])
        labels = list(all_labels[split_idx:])

    # Map function: decode image (no resize) + one-hot label
    def load_image(path, label):
        raw = tf.io.read_file(path)
        img = tf.image.decode_image(raw, channels=3, expand_animations=False)
        img = tf.cast(img, tf.float32)            # pixels stay in [0, 255]
        if label_mode == "categorical":
            label = tf.one_hot(label, num_classes)
        return img, label

    ds = tf.data.Dataset.from_tensor_slices((files, labels))
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.padded_batch(batch_size)              # handles variable image sizes
    ds.class_names = class_names                  # expose class names like Keras does
    return ds


train_ds = build_dataset_no_resize(
    DataSet_Dir, validation_split=0.2, subset="training",
    seed=123, batch_size=batch_size, label_mode="categorical")

print(f"Train batches : {tf.data.experimental.cardinality(train_ds).numpy()}")
print(f"Classes       : {train_ds.class_names}")


Train batches : 342
Classes       : ['closed', 'no_yawn', 'open', 'other_activities', 'safe_driving', 'talking_phone', 'texting_phone', 'turning', 'yawn']


In [4]:
temp_ds = build_dataset_no_resize(
    DataSet_Dir, validation_split=0.2, subset="validation",
    seed=123, batch_size=batch_size, label_mode="categorical")

print(f"Temp batches  : {tf.data.experimental.cardinality(temp_ds).numpy()}")


Temp batches  : 86


In [5]:
# Count total batches in temp
temp_batches = tf.data.experimental.cardinality(temp_ds).numpy()
val_size     = temp_batches // 2       # first half  → validation

val_ds  = temp_ds.take(val_size)       # first 50% of temp
test_ds = temp_ds.skip(val_size)       # last  50% of temp

class_names = train_ds.class_names


In [13]:
# performance optimisation
AUTOTUNE  = tf.data.AUTOTUNE
train_ds  = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds    = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds   = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [7]:
#NOTEBOOK_DIR = "/content/drive/MyDrive/DriverSafety"
Output_Dir   = "./Data_Split"

Create class folders for given split

In [8]:
# ═══════════════════════════════════════════════════════════════════
#  METHOD 1 — Create class folders for a given split
# ═══════════════════════════════════════════════════════════════════
def create_split_folders(split_name: str,
                         class_names: list,
                         output_dir: Path) -> dict:
    """
    Creates:  output_dir / split_name / class_name /

    Parameters
    ----------
    split_name  : 'train' | 'val' | 'test'
    class_names : list of class name strings
    output_dir  : base Path  (e.g. Data_Split/)

    Returns
    -------
    folder_paths : { class_name : Path }
    """
    print(f"\n[{split_name}] Creating folders …")

    # ── Create the split folder first  (e.g. Data_Split/train/) ───
    split_folder = output_dir / split_name
    split_folder.mkdir(parents=True, exist_ok=True)
    print(f"  ✔  {split_folder}  ← split folder")

    # ── Create one class folder inside it per class ────────────────
    folder_paths = {}
    for class_name in class_names:
        folder = split_folder / class_name
        folder.mkdir(parents=True, exist_ok=True)
        folder_paths[class_name] = folder
        print(f"  ✔  {folder}")

    print(f"[{split_name}] 1 split folder + {len(class_names)} class folders created ✓")

    # ── Verify everything exists on disk ──────────────────────────
    print(f"\n[{split_name}] Verifying folders on disk …")
    all_ok = True

    # Check split folder
    if split_folder.exists():
        print(f"  ✔ EXISTS  →  {split_folder}")
    else:
        print(f"  ✘ MISSING →  {split_folder}")
        all_ok = False

    # Check each class folder
    for class_name, folder in folder_paths.items():
        if folder.exists():
            print(f"  ✔ EXISTS  →  {folder}")
        else:
            print(f"  ✘ MISSING →  {folder}")
            all_ok = False

    if all_ok:
        print(f"\n  ✅ All folders verified successfully!")
    else:
        print(f"\n  ⚠️  Some folders are missing — check Drive permissions")

    return folder_paths


In [9]:
import shutil

def copy_split_files(files_and_classes: list,
                     split_name: str,
                     output_dir) -> dict:
    """
    Copy image files AS-IS using shutil.copy2.
    No format conversion, no re-encoding — original bytes preserved.

    Parameters
    ----------
    files_and_classes : list of (Path, class_name_str) tuples
    split_name        : 'train' | 'val' | 'test'
    output_dir        : base Path  (e.g. Data_Split/)

    Returns
    -------
    counters : { class_name : int }
    """
    total    = len(files_and_classes)
    counters = {}
    print(f"[{split_name}] Copying {total} files …")

    for i, (src, cls) in enumerate(files_and_classes, 1):
        dst_folder = output_dir / split_name / cls
        dst_folder.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst_folder / src.name)   # copy bytes as-is
        counters[cls] = counters.get(cls, 0) + 1

        if i % 500 == 0 or i == total:
            pct = i / total * 100
            bar = "█" * int(pct // 5) + "░" * (20 - int(pct // 5))
            print(f"  [{bar}] {pct:5.1f}%  ({i}/{total})", end="")

    print(f"  [{'█'*20}] 100.0%  -- {split_name} complete!              ")
    return counters


In [10]:
# ═══════════════════════════════════════════════════════════════════
#  METHOD 3 — Print verification summary for one split
# ═══════════════════════════════════════════════════════════════════
def verify_split(split_name: str,
                 class_names: list,
                 folder_paths: dict) -> int:
    """
    Counts saved images in each class folder and prints a summary table.

    Returns total number of images saved in this split.
    """
    print(f"\n  {'─'*45}")
    print(f"  {split_name.upper():<10} {'Class':<20} {'Images':>8}")
    print(f"  {'─'*45}")

    total = 0
    for class_name in class_names:
        count  = len(list(folder_paths[class_name].glob("*.jpg")))
        total += count
        bar    = "▓" * (count // 10)
        print(f"  {'':<10} {class_name:<20} {count:>6}  {bar}")

    print(f"  {'─'*45}")
    print(f"  {'':<10} {'TOTAL':<20} {total:>8}")
    return total

In [11]:
# ═══════════════════════════════════════════════════════════════════
#  METHOD 4 — Master method: runs all 3 steps for one split
# ═══════════════════════════════════════════════════════════════════
def process_split(dataset: tf.data.Dataset,
                  split_name: str,
                  class_names: list,
                  output_dir: Path) -> dict:
    """
    Full pipeline for ONE split:
      1. Create class folders
      2. Save all images from dataset
      3. Verify counts

    Parameters
    ----------
    dataset     : tf.data.Dataset
    split_name  : 'train' | 'val' | 'test'
    class_names : list of class name strings
    output_dir  : base output Path

    Returns
    -------
    counters : { class_name : int }
    """
    print("\n" + "█"*55)
    print(f"  PROCESSING  →  {split_name.upper()}")
    print("█"*55)

    # Step 1 — Create folders
    folder_paths = create_split_folders(split_name, class_names, output_dir)

    # Step 2 — Save images
    counters     = save_dataset_images(dataset, split_name,
                                       folder_paths, class_names)

    # Step 3 — Verify
    total        = verify_split(split_name, class_names, folder_paths)

    print(f"\n  ✅ {split_name.upper()} — {total} images saved to Drive ✓")
    return counters

In [12]:
# ═══════════════════════════════════════════════════════════════════
#  METHOD 5 — Final summary across all 3 splits
# ═══════════════════════════════════════════════════════════════════
def print_final_summary(all_counters: dict,
                        class_names: list,
                        output_dir: Path) -> None:
    """
    Prints a combined table showing image counts across
    train / val / test for every class.

    all_counters : { 'train': {class: count}, 'val': ..., 'test': ... }
    """
    print("\n\n" + "═"*65)
    print("  FINAL SUMMARY — All Splits")
    print("═"*65)
    print(f"  {'Class':<20} {'Train':>8}  {'Val':>8}  {'Test':>8}  {'Total':>8}")
    print("─"*65)

    grand_total = 0
    split_sums  = {"train": 0, "val": 0, "test": 0}

    for class_name in class_names:
        t = all_counters["train"].get(class_name, 0)
        v = all_counters["val"].get(class_name, 0)
        s = all_counters["test"].get(class_name, 0)
        row_total    = t + v + s
        grand_total += row_total
        split_sums["train"] += t
        split_sums["val"]   += v
        split_sums["test"]  += s
        print(f"  {class_name:<20} {t:>8}  {v:>8}  {s:>8}  {row_total:>8}")

    print("─"*65)
    print(f"  {'TOTAL':<20} "
          f"{split_sums['train']:>8}  "
          f"{split_sums['val']:>8}  "
          f"{split_sums['test']:>8}  "
          f"{grand_total:>8}")
    print("═"*65)

    # Folder tree
    print(f"\n  Google Drive folder tree:")
    print(f"  MyDrive/ML_Project/")
    print(f"  └── Data_Split/")
    for split in ("train", "val", "test"):
        print(f"      ├── {split}/")
        for class_name in class_names:
            count = all_counters[split].get(class_name, 0)
            print(f"      │    ├── {class_name}/  ({count} images)")

    print(f"\n  📁 Saved to : {output_dir.resolve()}")
    print(f"  ✅ {grand_total} total images saved to Google Drive!\n")

In [15]:
import os, random, shutil
from pathlib import Path

# -- Configuration ------------------------------------------------------
INPUT_DIR  = Path(DataSet_Dir)
OUTPUT_DIR = Path("./Data_Split")
SEED       = 123
EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tiff", ".webp"}

# -- Discover classes and collect all file paths ------------------------
class_names = sorted([d.name for d in INPUT_DIR.iterdir() if d.is_dir()])
print(f"Classes ({len(class_names)}): {class_names}")

all_pairs = []   # list of (Path, class_name)
for cls in class_names:
    for f in sorted((INPUT_DIR / cls).iterdir()):
        if f.suffix.lower() in EXTENSIONS:
            all_pairs.append((f, cls))

print(f"Total images found : {len(all_pairs)}")

# -- Deterministic shuffle (same seed=123 as before) -------------------
random.seed(SEED)
random.shuffle(all_pairs)

# -- 80 / 10 / 10 split ------------------------------------------------
n         = len(all_pairs)
train_end = int(n * 0.80)
val_end   = int(n * 0.90)

splits = {
    "train" : all_pairs[:train_end],
    "val"   : all_pairs[train_end:val_end],
    "test"  : all_pairs[val_end:],
}
print(f"Split  ->  train: {len(splits['train'])}  val: {len(splits['val'])}  test: {len(splits['test'])}")

# -- Copy files as-is into Data_Split/split/class/ ---------------------
all_counters = {}
for split_name, pairs in splits.items():
    all_counters[split_name] = copy_split_files(pairs, split_name, OUTPUT_DIR)

# -- Final summary table -----------------------------------------------
SEP = "=" * 65
sep = "-" * 63
print(f"{SEP}")
print(f"  {'Class':<22} {'Train':>8}  {'Val':>8}  {'Test':>8}  {'Total':>8}")
print(f"  {sep}")
for cls in class_names:
    t = all_counters["train"].get(cls, 0)
    v = all_counters["val"].get(cls, 0)
    s = all_counters["test"].get(cls, 0)
    print(f"  {cls:<22} {t:>8}  {v:>8}  {s:>8}  {t+v+s:>8}")
print(f"  {sep}")
grand  = sum(c for sp in all_counters.values() for c in sp.values())
tr_tot = sum(all_counters["train"].values())
v_tot  = sum(all_counters["val"].values())
ts_tot = sum(all_counters["test"].values())
print(f"  {'TOTAL':<22} {tr_tot:>8}  {v_tot:>8}  {ts_tot:>8}  {grand:>8}")
print(f"{SEP}")
print(f"Saved to : {OUTPUT_DIR.resolve()}")
print("Original formats preserved -- no re-encoding performed.")


Classes (9): ['closed', 'no_yawn', 'open', 'other_activities', 'safe_driving', 'talking_phone', 'texting_phone', 'turning', 'yawn']
Total images found : 13651
Split  ->  train: 10920  val: 1365  test: 1366
[train] Copying 10920 files …
  [░░░░░░░░░░░░░░░░░░░░]   4.6%  (500/10920)  [█░░░░░░░░░░░░░░░░░░░]   9.2%  (1000/10920)  [██░░░░░░░░░░░░░░░░░░]  13.7%  (1500/10920)  [███░░░░░░░░░░░░░░░░░]  18.3%  (2000/10920)  [████░░░░░░░░░░░░░░░░]  22.9%  (2500/10920)  [█████░░░░░░░░░░░░░░░]  27.5%  (3000/10920)  [██████░░░░░░░░░░░░░░]  32.1%  (3500/10920)  [███████░░░░░░░░░░░░░]  36.6%  (4000/10920)  [████████░░░░░░░░░░░░]  41.2%  (4500/10920)  [█████████░░░░░░░░░░░]  45.8%  (5000/10920)  [██████████░░░░░░░░░░]  50.4%  (5500/10920)  [██████████░░░░░░░░░░]  54.9%  (6000/10920)  [███████████░░░░░░░░░]  59.5%  (6500/10920)  [████████████░░░░░░░░]  64.1%  (7000/10920)  [█████████████░░░░░░░]  68.7%  (7500/10920)  [██████████████░░░░░░]  73.3%  (8000/10920)  [███████████████░░░░░]  77.8%  (8500/10920)

In [16]:
from pathlib import Path

Output_Dir = Path("./Data_Split")

print(f"Data_Split exists : {Output_Dir.exists()}")
print(f"\nContents:")
if Output_Dir.exists():
    for item in sorted(Output_Dir.rglob("*")):
        if item.is_dir():
            n_files = len(list(item.glob("*.jpg")))
            print(f"  📁  {item.relative_to(Output_Dir)}  "
                  f"({n_files} images)")
else:
    print("  ✘  Data_Split folder not found on Drive")

Data_Split exists : True

Contents:
  📁  test  (0 images)
  📁  test\closed  (74 images)
  📁  test\no_yawn  (73 images)
  📁  test\open  (78 images)
  📁  test\other_activities  (193 images)
  📁  test\safe_driving  (222 images)
  📁  test\talking_phone  (222 images)
  📁  test\texting_phone  (207 images)
  📁  test\turning  (188 images)
  📁  test\yawn  (85 images)
  📁  train  (0 images)
  📁  train\closed  (586 images)
  📁  train\no_yawn  (567 images)
  📁  train\open  (576 images)
  📁  train\other_activities  (1652 images)
  📁  train\safe_driving  (1734 images)
  📁  train\talking_phone  (1716 images)
  📁  train\texting_phone  (1744 images)
  📁  train\turning  (1634 images)
  📁  train\yawn  (572 images)
  📁  val  (0 images)
  📁  val\closed  (66 images)
  📁  val\no_yawn  (85 images)
  📁  val\open  (72 images)
  📁  val\other_activities  (214 images)
  📁  val\safe_driving  (223 images)
  📁  val\talking_phone  (207 images)
  📁  val\texting_phone  (238 images)
  📁  val\turning  (174 images)
  📁  va